In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*ประมาณการการใช้งาน: ไม่ถึงหนึ่งนาทีบนโปรเซสเซอร์ Eagle r3 (หมายเหตุ: นี่เป็นเพียงการประมาณการเท่านั้น เวลาจริงอาจแตกต่างกัน)*

## พื้นฐาน

เพื่อให้ได้ผลลัพธ์ที่เร็วและมีประสิทธิภาพมากขึ้น ตั้งแต่วันที่ 1 มีนาคม 2024 เป็นต้นไป Circuit และ observable ต้องถูกแปลงให้ใช้คำสั่งที่ QPU (หน่วยประมวลผลควอนตัม) รองรับเท่านั้น ก่อนส่งไปยัง Qiskit Runtime primitives เราเรียกสิ่งเหล่านี้ว่า *instruction set architecture* (ISA) Circuit และ observable วิธีทั่วไปในการทำเช่นนี้คือใช้ฟังก์ชัน `generate_preset_pass_manager` ของ Transpiler อย่างไรก็ตาม คุณอาจเลือกทำตามกระบวนการแบบ manual ก็ได้

ตัวอย่างเช่น คุณอาจต้องการกำหนดเป้าหมายเป็น Qubit กลุ่มเฉพาะบนอุปกรณ์เฉพาะ บทเรียนนี้ทดสอบประสิทธิภาพของการตั้งค่า Transpiler ต่าง ๆ โดยผ่านกระบวนการสร้าง transpile และส่ง Circuit อย่างครบถ้วน

## ข้อกำหนดเบื้องต้น

ก่อนเริ่มต้น ตรวจสอบให้แน่ใจว่าติดตั้งสิ่งต่อไปนี้แล้ว:

* Qiskit SDK v1.2 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.28 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)

## การตั้งค่า

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

## ขั้นตอนที่ 1: แมปอินพุตคลาสสิกไปยังปัญหาควอนตัม
สร้าง Circuit ขนาดเล็กให้ Transpiler ลองเพิ่มประสิทธิภาพ ตัวอย่างนี้สร้าง Circuit ที่ทำ Grover's algorithm พร้อม oracle ที่ทำเครื่องหมายสถานะ `111` จากนั้น จำลองการแจกแจงในอุดมคติ (สิ่งที่คาดว่าจะวัดได้หากรันบนคอมพิวเตอร์ควอนตัมที่สมบูรณ์แบบจำนวนครั้งอนันต์) เพื่อเปรียบเทียบในภายหลัง

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

## Transpile

Next, transpile the circuits for the QPU. You will compare the performance of the transpiler with `optimization_level` set to `0` (lowest) against `3` (highest). The lowest optimization level does the bare minimum needed to get the circuit running on the device; it maps the circuit qubits to the device qubits and adds swap gates to allow all two-qubit operations. The highest optimization level is much smarter and uses lots of tricks to reduce the overall gate count. Since multi-qubit gates have high error rates and qubits decohere over time, the shorter circuits should give better results.

<Admonition type="important">
    This example uses IBM Quantum&reg; hardware, but you can try it on any Qiskit-compatible QPU.  Your results might be different.
</Admonition>

The following cell transpiles `qc` for both values of `optimization_level`, prints the number of two-qubit gates, and adds the transpiled circuits to a list. Some of the transpiler's algorithms are randomized, so it sets a seed for reproducibility.

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

## Execute the circuit

At this point, you have a list of circuits transpiled with different settings. Next, run these circuits using the Sampler primitive and store the results to `result`.

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif)

## ขั้นตอนที่ 3: รันโดยใช้ Qiskit primitives
ณ จุดนี้ คุณมีรายการ Circuit ที่ transpile แล้วสำหรับ QPU ที่กำหนด ต่อไป สร้าง instance ของ Sampler primitive และเริ่ม batched job โดยใช้ context manager (`with ...:`) ซึ่งจะเปิดและปิด batch โดยอัตโนมัติ

ภายใน context manager ให้ sample Circuit และเก็บผลลัพธ์ไว้ใน `result`

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

## ขั้นตอนที่ 4: ประมวลผลหลังการรันและคืนผลลัพธ์ในรูปแบบคลาสสิกที่ต้องการ
สุดท้าย plot ผลลัพธ์จากการรันบนอุปกรณ์เทียบกับการแจกแจงในอุดมคติ คุณจะเห็นว่าผลลัพธ์ที่ได้ด้วย `optimization_level=3` ใกล้เคียงกับการแจกแจงในอุดมคติมากกว่า เนื่องจากจำนวน Gate น้อยกว่า และ `optimization_level=3 + dd` ใกล้เคียงยิ่งขึ้นไปอีกเนื่องจาก dynamic decoupling

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/525777ea-d438-4f3b-acb6-53e579f24a0e-0.avif)

คุณสามารถยืนยันสิ่งนี้ได้โดยคำนวณ [Hellinger fidelity](https://docs.quantum.ibm.com/api/qiskit/quantum_info) ระหว่างผลลัพธ์แต่ละชุดกับการแจกแจงในอุดมคติ (ค่าสูงกว่าดีกว่า และ 1 คือ fidelity สมบูรณ์แบบ)